# Week 8 - Day 3: Vector Databases and Semantic Search

## Learning Objectives

By the end of this session, I will be able to:

1. Explain what a vector database is.
2. Understand how embeddings are stored and searched.
3. Explain semantic search.
4. Understand cosine similarity and distance.
5. Store document chunks and metadata in ChromaDB.
6. Convert a user question into an embedding.
7. Retrieve relevant document chunks using semantic search.
8. Understand how retrieval supports a Chat with PDF application.

## 1. Quick Review: What Is an Embedding?

An embedding is a numerical vector representation of data,
such as text.

Example:

Text:
"I enjoy learning Python."

Embedding:
[0.12, -0.34, 0.56, ...]

The embedding model converts text into a numerical
representation that can be compared mathematically.

In RAG, embeddings help connect a user's question
with related document chunks.

## 2. Review: Document Chunking

In Day 2, we learned that chunking splits a large
document into smaller pieces.

Example:

PDF
 ↓
Extracted Text
 ↓
Chunk 1
Chunk 2
Chunk 3
...

Each chunk can be converted into an embedding.

The chunk text and its metadata should be preserved
so that retrieved information can be traced back
to its source.

In [1]:
# Simple example of document chunks

document_chunks = [
    "Employees work from Monday to Friday.",
    "Employees receive 12 casual leave days per year.",
    "Employees may work remotely up to two days per week."
]

for index, chunk in enumerate(document_chunks, start=1):
    print(f"Chunk {index}: {chunk}")

Chunk 1: Employees work from Monday to Friday.
Chunk 2: Employees receive 12 casual leave days per year.
Chunk 3: Employees may work remotely up to two days per week.


## 3. What Is a Vector Database?

A vector database is a system designed to store,
organize, and search vector representations.

In an embedding-based RAG system, it can store:

- Embedding vectors
- Associated document text or references
- Metadata
- Unique identifiers

When a user asks a question, the system can search
for vectors that are similar to the question vector.

The matching records can then be used to retrieve
relevant document chunks.

## 4. Why Do We Need a Vector Database?

Imagine a company has 10,000 PDF documents.

Each document has been divided into chunks.

Each chunk has an embedding.

Searching through every chunk manually would be
slow and impractical.

A vector database helps organize and search these
vector representations.

Depending on the database and configuration, it
may use indexing methods to make similarity search
more efficient.

## 5. What Does a Vector Database Store?

A record in a vector database may contain:

1. ID
   - A unique identifier for the record.

2. Embedding
   - The numerical vector representation.

3. Text or Document
   - The original chunk or a reference to it.

4. Metadata
   - Source filename, page number, category, etc.

Example:

ID: chunk_001

Text:
"Employees receive 12 casual leave days per year."

Embedding:
[0.21, -0.13, 0.67, ...]

Metadata:
{
    "source": "employee_handbook.pdf",
    "page": 2
}

## 6. Vector Database vs Traditional Database

### Traditional Database

Often stores structured information such as:

- Employee ID
- Employee name
- Department
- Salary

Queries may filter or match records using fields
and conditions.

### Vector Database

Stores or indexes vector representations and supports
similarity-based retrieval.

It can help find semantically related text even when
the wording is different.

Important:
Vector databases and traditional databases solve
different problems, but they can also be used together.

## 7. Example: Traditional Search vs Vector Search

Suppose a document contains:

"Employees may work remotely up to two days per week."

User question:

"How often can I work from home?"

A keyword search may struggle if it relies on exact
word matches.

A semantic search system can compare embeddings and
may retrieve the remote-work passage because the
meanings are related.

Retrieval depends on the embedding model, stored data,
and search configuration.

## 8. What Is ChromaDB?

ChromaDB, commonly called Chroma, is a database
designed for working with embeddings and retrieval.

It can store documents, embeddings, and metadata,
and supports similarity search.

We will use ChromaDB for our learning exercise.

The purpose is to understand the vector database
workflow, not to build a production system yet.

In [2]:
%pip install -U chromadb sentence-transformers

   ---------------------------------------- 0.0/740.6 kB ? eta -:--:--
   ---------------------------------------- 740.6/740.6 kB 4.3 MB/s  0:00:00
  Attempting uninstall: sentence-transformers
    Found existing installation: sentence-transformers 6.0.1
    Uninstalling sentence-transformers-6.0.1:
      Successfully uninstalled sentence-transformers-6.0.1
Note: you may need to restart the kernel to use updated packages.


## 9. Import the Required Libraries

We will use:

- ChromaDB for vector storage and retrieval
- Sentence Transformers for generating embeddings

We will use a sentence embedding model to represent
document chunks and user questions as vectors.

In [3]:
import chromadb

from sentence_transformers import SentenceTransformer

print("Libraries imported successfully.")

C:\Users\LENOVO\miniconda3\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully.


## 10. Load the Embedding Model

We will use:

sentence-transformers/all-MiniLM-L6-v2

This model converts text into fixed-size embeddings.

The same embedding model should be used consistently
for document chunks and user questions in this exercise.

The first model download may take some time.

In [4]:
embedding_model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)

print("Embedding model loaded.")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1183.61it/s]


Embedding model loaded.


## 11. Create a Small Knowledge Base

We will create a small collection of employee
handbook information.

Each piece of text represents a document chunk.

Our sample data includes:

- Working hours
- Leave policy
- Remote work
- Salary payment
- IT security

Later, we will search this collection using
natural-language questions.

In [5]:
documents = [
    "Employees work from Monday to Friday.",
    "Standard working hours are 9:00 AM to 6:00 PM.",
    "Employees receive 12 casual leave days per year.",
    "Leave requests must be approved by a manager.",
    "Employees may work remotely up to two days per week.",
    "Salary is paid on the last working day of each month.",
    "Employees must use strong passwords to protect company accounts.",
    "The IT team helps employees resolve technical issues."
]

print("Number of documents:", len(documents))

Number of documents: 8


In [6]:
# Display all document chunks

for index, document in enumerate(documents, start=1):
    print(f"Document {index}: {document}")

Document 1: Employees work from Monday to Friday.
Document 2: Standard working hours are 9:00 AM to 6:00 PM.
Document 3: Employees receive 12 casual leave days per year.
Document 4: Leave requests must be approved by a manager.
Document 5: Employees may work remotely up to two days per week.
Document 6: Salary is paid on the last working day of each month.
Document 7: Employees must use strong passwords to protect company accounts.
Document 8: The IT team helps employees resolve technical issues.


## 12. Add Metadata to Documents

Metadata describes the document chunk.

For this exercise, we will store:

- Source filename
- Page number
- Category

In a real PDF pipeline, metadata would be derived
from the actual uploaded document and its structure.

Here, the values are sample metadata created
for learning.

In [7]:
metadatas = [
    {"source": "employee_handbook.pdf", "page": 1, "category": "working_hours"},
    {"source": "employee_handbook.pdf", "page": 1, "category": "working_hours"},
    {"source": "employee_handbook.pdf", "page": 2, "category": "leave"},
    {"source": "employee_handbook.pdf", "page": 2, "category": "leave"},
    {"source": "employee_handbook.pdf", "page": 3, "category": "remote_work"},
    {"source": "employee_handbook.pdf", "page": 4, "category": "salary"},
    {"source": "employee_handbook.pdf", "page": 5, "category": "security"},
    {"source": "employee_handbook.pdf", "page": 6, "category": "IT_support"}
]

print("Metadata records:", len(metadatas))

Metadata records: 8


## 13. Convert Document Chunks into Embeddings

Each document chunk needs a vector representation
for embedding-based similarity search.

We will use the embedding model loaded earlier.

The result is one vector per document chunk.

The vectors can then be stored in the vector database.

In [8]:
document_embeddings = embedding_model.encode(documents)

print("Embedding shape:", document_embeddings.shape)

Embedding shape: (8, 384)


### Understanding the Output

The output shape contains:

(number of document chunks, embedding dimensions)

Each row represents one document chunk.

Each row contains the numerical embedding generated
by the model.

In [9]:
# Inspect the first document embedding

print(document_embeddings[0])

print("Dimensions:", len(document_embeddings[0]))

[-2.72640213e-02 -1.38868103e-02  3.94203551e-02 -2.07914449e-02
 -1.87759791e-02  2.28139833e-02 -3.05347182e-02 -1.02890112e-01
 -1.18078664e-01 -3.94610912e-02 -7.31454603e-03  9.28576440e-02
 -9.78417322e-02  5.78198731e-02  2.83953547e-02 -7.80286342e-02
 -4.67610098e-02 -5.11535583e-03 -1.95632745e-02 -6.41737059e-02
 -2.96133757e-02 -8.47677216e-02 -6.88970983e-02  4.67794463e-02
  2.95046568e-02  9.49176848e-02  6.95582340e-03 -1.08679458e-02
 -4.35163416e-02 -1.73773221e-03 -1.23088919e-01 -6.14872500e-02
 -3.09224296e-02  3.03324722e-02 -2.11494155e-02  5.60613386e-02
 -1.04740597e-02  4.73045595e-02  5.46760904e-03  3.74063849e-02
 -4.10965644e-02  9.71451402e-03 -4.63625137e-03  3.70203890e-02
 -8.77590701e-02 -1.58453155e-02  5.90658449e-02  3.61270308e-02
  2.20494550e-02  7.65096545e-02  7.65366554e-02 -4.90807667e-02
  3.45067456e-02  5.31568453e-02  6.30695745e-02  2.84820963e-02
  3.43785784e-03 -1.12001501e-01  6.94568753e-02 -1.53689943e-02
 -2.32596672e-03 -2.38125

### Observation

The first document chunk has been converted into
a numerical vector.

We will store these vectors with their corresponding
document text and metadata.

## 14. Create a ChromaDB Client

A client provides access to ChromaDB.

For this notebook, we will use an in-memory client.

This is convenient for learning, but the data will
not persist after the client session ends.

Persistent storage will be covered in a later
implementation or project step.

In [10]:
client = chromadb.EphemeralClient()

print("ChromaDB client created.")

ChromaDB client created.


## 15. Create a Collection

A collection groups related records.

For our example, we will create a collection
named employee_handbook.

The collection will contain our document chunks
and their embeddings.

In [11]:
collection = client.create_collection(
    name="employee_handbook"
)

print("Collection created.")

Collection created.


## 16. Add Documents to the Collection

We will add:

- Unique IDs
- Document text
- Metadata
- Precomputed embeddings

Each record will represent one document chunk.

The IDs allow us to identify individual records.

In [12]:
document_ids = [
    f"chunk_{index:03d}"
    for index in range(1, len(documents) + 1)
]

collection.add(
    ids=document_ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=document_embeddings.tolist()
)

print("Documents added to ChromaDB.")

Documents added to ChromaDB.


### Observation

Our collection now contains document records.

Each record has:

- An ID
- Text
- Metadata
- An embedding

We can now search the collection using a user question.

In [13]:
print("Total records in collection:", collection.count())

Total records in collection: 8


## 17. What Is Semantic Search?

Semantic search retrieves information based on
meaning rather than relying only on exact keyword matches.

In an embedding-based system:

1. Convert the query into an embedding.
2. Compare it with stored document embeddings.
3. Retrieve the most similar records.

This can help match different phrases that express
related meanings.

Example:

Query:
"How often can I work from home?"

Document:
"Employees may work remotely up to two days per week."

The system may retrieve the remote-work document
because the query and document are semantically related.

## 18. Search Query vs Document Embeddings

In our collection:
```
Document chunk
      ↓
Embedding model
      ↓
Stored document vector

User question
      ↓
Same embedding model
      ↓
Question vector
```
The database compares the question vector with
stored document vectors and retrieves matches.

In [14]:
user_question = "How often can I work from home?"

question_embedding = embedding_model.encode(
    user_question
)

print("Question:", user_question)
print("Embedding dimensions:", len(question_embedding))

Question: How often can I work from home?
Embedding dimensions: 384


## 19. Query ChromaDB

We will search the collection using the question
embedding.

The n_results parameter controls how many results
we request.

For this example, we will retrieve the top 3 matches.

The database returns matching records and distance
values according to its configured distance metric.

In [15]:
results = collection.query(
    query_embeddings=[question_embedding.tolist()],
    n_results=3
)

print(results)

{'ids': [['chunk_005', 'chunk_003', 'chunk_001']], 'embeddings': None, 'documents': [['Employees may work remotely up to two days per week.', 'Employees receive 12 casual leave days per year.', 'Employees work from Monday to Friday.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[{'source': 'employee_handbook.pdf', 'page': 3, 'category': 'remote_work'}, {'page': 2, 'category': 'leave', 'source': 'employee_handbook.pdf'}, {'category': 'working_hours', 'page': 1, 'source': 'employee_handbook.pdf'}]], 'distances': [[0.8875924944877625, 1.141363263130188, 1.1541757583618164]]}


### Understanding the Search Results

The query result may include:

- IDs
- Documents
- Metadata
- Distances

The returned documents are candidate matches for
the user's question.

A distance value is not automatically a probability
or a guarantee of correctness.

Its interpretation depends on the distance metric
and database configuration.

In [16]:
# Display the retrieved documents clearly

retrieved_documents = results["documents"][0]
retrieved_ids = results["ids"][0]
retrieved_metadata = results["metadatas"][0]
retrieved_distances = results["distances"][0]

for i in range(len(retrieved_documents)):
    print(f"Result {i + 1}")
    print("ID:", retrieved_ids[i])
    print("Text:", retrieved_documents[i])
    print("Metadata:", retrieved_metadata[i])
    print("Distance:", retrieved_distances[i])
    print("-" * 50)

Result 1
ID: chunk_005
Text: Employees may work remotely up to two days per week.
Metadata: {'source': 'employee_handbook.pdf', 'page': 3, 'category': 'remote_work'}
Distance: 0.8875924944877625
--------------------------------------------------
Result 2
ID: chunk_003
Text: Employees receive 12 casual leave days per year.
Metadata: {'page': 2, 'category': 'leave', 'source': 'employee_handbook.pdf'}
Distance: 1.141363263130188
--------------------------------------------------
Result 3
ID: chunk_001
Text: Employees work from Monday to Friday.
Metadata: {'category': 'working_hours', 'page': 1, 'source': 'employee_handbook.pdf'}
Distance: 1.1541757583618164
--------------------------------------------------


### Observation

Review the retrieved chunks.

Check whether the remote-work policy appears
among the results.

The exact ordering and distance values depend on
the embedding model and database configuration.

Semantic search is a retrieval method, not a guarantee
that every returned chunk answers the question.

## 20. Test Multiple User Questions

A useful retrieval system should work across
different questions.

We will test questions about:

- Leave
- Salary
- Working hours
- IT support

Observe which document chunks are retrieved
for each question.

In [17]:
questions = [
    "How many casual leaves do employees get?",
    "When is the salary paid?",
    "What are the normal office hours?",
    "Who can help me with a technical issue?"
]

for question in questions:
    query_embedding = embedding_model.encode(question)

    query_results = collection.query(
        query_embeddings=[query_embedding.tolist()],
        n_results=2
    )

    print("\nQuestion:", question)

    for document in query_results["documents"][0]:
        print("-", document)


Question: How many casual leaves do employees get?
- Employees receive 12 casual leave days per year.
- Employees work from Monday to Friday.

Question: When is the salary paid?
- Salary is paid on the last working day of each month.
- Standard working hours are 9:00 AM to 6:00 PM.

Question: What are the normal office hours?
- Standard working hours are 9:00 AM to 6:00 PM.
- Employees may work remotely up to two days per week.

Question: Who can help me with a technical issue?
- The IT team helps employees resolve technical issues.
- Employees must use strong passwords to protect company accounts.


### Observation

Check whether the retrieved chunks match the
topic of each question.

Ask yourself:

1. Did the system retrieve relevant information?
2. Did it retrieve unrelated information?
3. Did the top result contain enough information?
4. Would more than two results help?

Retrieval quality should be tested with multiple
questions, not just one successful example.

## 21. Create a Reusable Search Function

Instead of writing the same query code repeatedly,
we can create a function.

The function will:

1. Accept a user question.
2. Generate its embedding.
3. Query ChromaDB.
4. Return matching documents and metadata.

This is a basic retrieval component for our
Chat with PDF application.

In [18]:
def retrieve_documents(question, n_results=3):
    question_embedding = embedding_model.encode(question)

    results = collection.query(
        query_embeddings=[question_embedding.tolist()],
        n_results=n_results
    )

    retrieved = []

    for i, document in enumerate(results["documents"][0]):
        retrieved.append({
            "text": document,
            "metadata": results["metadatas"][0][i],
            "distance": results["distances"][0][i]
        })

    return retrieved

In [19]:
question = "What is the company's leave policy?"

retrieved = retrieve_documents(question, n_results=3)

for item in retrieved:
    print("Text:", item["text"])
    print("Metadata:", item["metadata"])
    print("Distance:", item["distance"])
    print("-" * 40)

Text: Leave requests must be approved by a manager.
Metadata: {'page': 2, 'source': 'employee_handbook.pdf', 'category': 'leave'}
Distance: 0.7408730387687683
----------------------------------------
Text: Employees receive 12 casual leave days per year.
Metadata: {'category': 'leave', 'source': 'employee_handbook.pdf', 'page': 2}
Distance: 0.8765595555305481
----------------------------------------
Text: Employees work from Monday to Friday.
Metadata: {'category': 'working_hours', 'page': 1, 'source': 'employee_handbook.pdf'}
Distance: 1.27296781539917
----------------------------------------


### Observation

We now have a reusable retrieval function.

It returns document text, metadata, and distance
for each retrieved result.

This function does not generate a final answer.
It only retrieves potentially relevant information.

The LLM answer-generation stage comes later.

## 22. Retrieval Is Not the Same as Generation

Retrieval and generation are separate tasks.

### Retrieval

Finds relevant information from the knowledge base.

### Generation

Uses the question and available context to produce
a natural-language response.

A RAG system combines both.
```
User Question
      ↓
Semantic Search
      ↓
Relevant Chunks
      ↓
Context Injection
      ↓
LLM
      ↓
Generated Answer

## 23. Example: Retrieved Context

Question:

"How often can I work from home?"

Retrieved chunk:

"Employees may work remotely up to two days per week,
subject to manager approval."

This retrieved chunk can be supplied to an LLM
as context.

The LLM can then generate a concise answer based
on the provided information.

The retrieval component does not itself write
the final natural-language answer.

In [21]:
question = "How often can I work from home?"

retrieved = retrieve_documents(question, n_results=2)

context = "\n\n".join(
    item["text"]
    for item in retrieved
)

print("Question:")
print(question)

print("\nRetrieved Context:")
print(context)

Question:
How often can I work from home?

Retrieved Context:
Employees may work remotely up to two days per week.

Employees receive 12 casual leave days per year.


### Observation

We have constructed a context string from the
retrieved document chunks.

This is a basic example of context preparation.

In the next lesson, we will study how retrieved
context is combined with a prompt and passed
to an LLM.

## 24. Important Retrieval Concepts

### Top-K Retrieval
Retrieve the K most relevant results according
to the search configuration.

### Similarity Search
Find vectors that are similar to a query vector.

### Distance
A numerical measure used to compare vectors.
Lower or higher values can indicate closer matches
depending on the metric.

### Metadata Filtering
Restrict search results using metadata conditions,
such as source or category.

### Retrieval Quality
How well the retrieved information supports
the user's question.

## 25. Common Limitations

### 1. Wrong Retrieval
The system may retrieve a related but incorrect passage.

### 2. Missing Information
The PDF may not contain the answer.

### 3. Poor Text Extraction
A PDF parser may fail to extract important content.

### 4. Poor Chunking
Relevant information may be split across chunks.

### 5. Embedding Limitations
The embedding model may not represent some relationships
well enough for the search task.

### 6. Too Many Results
Retrieving many irrelevant chunks can add noise
to the context supplied to the LLM.

RAG quality depends on the entire pipeline,
not only the vector database.

## 26. Vector Database vs Embedding Model

These are different components.

### Embedding Model
Converts text into numerical vectors.

### Vector Database
Stores and searches vector representations and
associated records.

### Example
```
Text
 ↓
Embedding Model
 ↓
Vector
 ↓
Vector Database
 ↓
Similarity Search
 ↓
Retrieved Document

## 27. Practice Exercises

### Exercise 1: Add Documents
Add three new document chunks to the knowledge base.

### Exercise 2: Generate Embeddings
Generate embeddings for the new chunks.

### Exercise 3: Store Records
Add the new chunks, IDs, and metadata to a collection.

### Exercise 4: Semantic Search
Ask a question using wording different from the
document wording.

### Exercise 5: Compare Results
Compare the top 1, top 3, and top 5 retrieved chunks.

### Exercise 6: Metadata
Print the source and page number for each result.

### Exercise 7: Reflection
Explain why a high similarity score does not
guarantee that a retrieved passage is correct.

## 28. Interview Questions

### Q1. What is a vector database?

A vector database stores and searches vector
representations, often supporting similarity-based
retrieval.

### Q2. What is semantic search?

Semantic search retrieves information based on
similarity in meaning rather than relying only
on exact keyword matches.

### Q3. What is the role of embeddings in semantic search?

Embeddings represent queries and documents as vectors
that can be compared mathematically.

### Q4. What is top-K retrieval?

Top-K retrieval requests the K highest-ranked
matches according to the search system's ranking
or distance configuration.

### Q5. What is metadata filtering?

Metadata filtering restricts records using
associated fields such as source, category,
or page number.

### Q6. Is a vector database the same as an embedding model?

No. The embedding model creates vectors. The vector
database stores and searches vectors and related records.

### Q7. Does semantic search always return the correct answer?

No. It returns candidate matches. The retrieved
content may be incomplete or irrelevant.

### Q8. What is the role of retrieval in RAG?

Retrieval finds relevant external information
that can be supplied to the LLM as context.

### Q9. What happens after retrieval?

The retrieved text is prepared as context and
provided to the LLM, which generates a response.

### Q10. Why preserve metadata?

Metadata helps identify the source of retrieved
information and supports filtering and traceability.

## 29. Day 3 Cheat Sheet

| Concept | Meaning |
|---|---|
| Embedding | Numerical representation of data |
| Vector database | Stores and searches vectors |
| Semantic search | Retrieval based on meaning |
| Similarity search | Finds vectors close to a query |
| Top-K | Number of results requested |
| Metadata | Information describing a record |
| Retrieval | Finding relevant information |
| Context injection | Supplying retrieved text to the LLM |

### Today's RAG Flow
```
Document Chunks
      ↓
Embedding Model
      ↓
Vector Database

User Question
      ↓
Question Embedding
      ↓
Semantic Search
      ↓
Relevant Chunks
      ↓
Context for LLM